<a href="https://colab.research.google.com/github/ganja025/ganja025/blob/main/iacolab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive
import subprocess
import time
import re
import threading
import io
from pyngrok import ngrok, conf
from google.colab import userdata
import requests
import shlex # Import shlex for proper shell quoting

# --- 1. Montar Drive ---
print("Montando Google Drive...")
drive.mount('/content/drive')

# --- 2. Rutas persistentes ---
CARPETA_DRIVE = "/content/drive/MyDrive/Colab Notebooks/llm_setup"
CARPETA_BIN = f"{CARPETA_DRIVE}/bin"
MODELO_NOMBRE = "qwen2.5-7b-instruct-q4_k_m.gguf"
MODEL_PATH_ORIGINAL = f"{CARPETA_DRIVE}/{MODELO_NOMBRE}"
URL_DESCARGA = "https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q4_K_M.gguf"

# Crear carpetas si no existen
os.makedirs(CARPETA_BIN, exist_ok=True)

# --- 3. Descargar Modelo si no existe ---
if not os.path.exists(MODEL_PATH_ORIGINAL):
    print("Descargando modelo a Drive...")
    !wget -O "{MODEL_PATH_ORIGINAL}" {URL_DESCARGA}
else:
    print("Modelo ya presente en Drive.")

# --- Crear enlace simbólico para el modelo sin espacios en la ruta ---
MODEL_PATH_SYMLINK = f"/tmp/{MODELO_NOMBRE}"
if os.path.exists(MODEL_PATH_SYMLINK):
    os.remove(MODEL_PATH_SYMLINK) # Eliminar si ya existe
os.symlink(MODEL_PATH_ORIGINAL, MODEL_PATH_SYMLINK)
print(f"Enlace simbólico creado: {MODEL_PATH_SYMLINK} -> {MODEL_PATH_ORIGINAL}")


# --- Función para terminar procesos existentes ---
def kill_process(name):
    escaped_name = re.escape(name)
    pids_output = subprocess.run(['pgrep', '-f', escaped_name], capture_output=True, text=True).stdout.strip()
    pids = pids_output.split('\n') if pids_output else []
    for pid in pids:
        if pid.isdigit():
            print(f"Terminando proceso '{name}' con PID {pid}...")
            try:
                subprocess.run(['kill', '-9', pid], check=True)
                time.sleep(1) # Dar tiempo para que el proceso termine
            except subprocess.CalledProcessError as e:
                print(f"Error al terminar el proceso {pid}: {e}")

# --- 4. Compilar O RECUPERAR el ejecutable llama-server ---
llama_server_local_path = "/content/llama-server"

# Asegurarse de que cualquier instancia de llama-server esté terminada antes de copiar el ejecutable
kill_process(llama_server_local_path)

if not os.path.exists(f"{CARPETA_BIN}/llama-server"):
    print("Compilando llama.cpp (esto solo pasará una vez si no está en Drive)...")
    !git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !cmake -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -S /content/llama.cpp
    !cmake --build /content/llama.cpp/build --config Release -j $(nproc)
    !cp /content/llama.cpp/build/bin/llama-server "{CARPETA_BIN}/llama-server"
    print("Binario llama-server guardado en Drive.")
    !cp "{CARPETA_BIN}/llama-server" {llama_server_local_path}
    !chmod +x {llama_server_local_path}
else:
    print("Ejecutable llama-server encontrado en Drive. Copiando a entorno local...")
    !cp "{CARPETA_BIN}/llama-server" {llama_server_local_path}
    !chmod +x {llama_server_local_path}

# --- 5. Lanzar servidor llama.cpp en background de forma robusta y esperar a que esté listo ---
print("\n--- Iniciando Servidor Llama.cpp ---")

# Terminar cualquier instancia anterior de llama-server si aún existiera
kill_process(llama_server_local_path)

# Directorio para los logs de nohup
LOG_DIR = "/tmp/llama_logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, "llama-server.log")

# Comando para iniciar llama-server usando el enlace simbólico sin espacios
llama_command = (
    f"nohup {llama_server_local_path} "
    f"-m {shlex.quote(MODEL_PATH_SYMLINK)} -ngl 99 --host 0.0.0.0 --port 8889 --temp 0.1 "
    f"> {LOG_FILE} 2>&1 &"
)

print(f"Iniciando llama-server en segundo plano con nohup. Logs en: {LOG_FILE}")
subprocess.Popen(llama_command, shell=True)

# Monitorear el archivo de log para saber cuándo el servidor está listo
server_ready = False
server_timeout = 180 # Tiempo de espera
check_interval = 2
server_start_time = time.time()

print(f"Esperando a que el servidor llama.cpp esté listo (hasta {server_timeout} segundos)... ")
while time.time() - server_start_time < server_timeout:
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r', encoding='utf-8', errors='ignore') as f:
            log_content = f.read()
            if "listening on http://0.0.0.0:8889" in log_content:
                server_ready = True
                print("\nServidor llama.cpp listo y escuchando en el puerto 8889.")
                break
    time.sleep(check_interval)

if not server_ready:
    print("\nERROR CRÍTICO: El servidor llama.cpp no se inició correctamente o no se puso a escuchar a tiempo.")
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r', encoding='utf-8', errors='ignore') as f:
            print("\n--- Contenido del Log de LLAMA.cpp ---\n" + f.read())
    raise RuntimeError("Llama.cpp server failed to start within timeout.")

# --- 6. Configurar y lanzar Ngrok ---
print("\n--- Configurando Ngrok ---")
!pip install pyngrok -qq # Instalar pyngrok silenciosamente

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if not NGROK_AUTH_TOKEN:
    print("ERROR: NGROK_AUTH_TOKEN no encontrado en los secretos de Colab. Por favor, añádelo para usar ngrok.")
    # Podemos salir o levantar un error aquí si ngrok es crítico
    raise RuntimeError("NGROK_AUTH_TOKEN is missing.")
else:
    print("Autenticando ngrok...")
    conf.get_default().auth_token = NGROK_AUTH_TOKEN

    # Cerrar cualquier túnel ngrok existente para evitar conflictos
    ngrok.kill()

    # Abrir un túnel HTTP a la dirección local del servidor llama.cpp
    try:
        print("Creando túnel ngrok para el puerto 8889...")
        public_url = ngrok.connect(8889)
        tunnel_url = public_url.public_url
        print(f"\n¡Túnel ngrok establecido con éxito! Tu URL pública es: {tunnel_url}")

    except Exception as e:
        print(f"ERROR al crear el túnel ngrok: {e}")
        print("Por favor, verifica lo siguiente:")
        print("1. Que el NGROK_AUTH_TOKEN sea válido.")
        print("2. Que el servidor llama.cpp esté realmente funcionando en el puerto 8889.")
        print("3. Puedes intentar reiniciar el entorno de Colab si los problemas persisten.")
        raise RuntimeError("Ngrok tunnel failed to establish.")

# --- 7. Realizar una solicitud de prueba al servidor LLM ---
print("\n--- Realizando solicitud de prueba al servidor LLM ---")
if 'tunnel_url' not in locals() or tunnel_url is None:
    print("Error: La URL del túnel no se ha encontrado. No se puede realizar la prueba.")
else:
    print(f"Intentando conectar al servidor LLM a través de: {tunnel_url}")
    try:
        generate_endpoint = f"{tunnel_url}/completion"

        headers = {
            "Content-Type": "application/json"
        }

        data = {
            "prompt": "Hola, ¿cómo estás?",
            "n_predict": 100, # Número máximo de tokens a generar
            "temperature": 0.7
        }

        response = requests.post(generate_endpoint, headers=headers, json=data, timeout=60) # Aumentar timeout
        response.raise_for_status() # Lanza una excepción para códigos de estado de error (4xx o 5xx)

        result = response.json()
        print("\n--- Respuesta del Servidor LLM ---")
        print(result.get('content', 'No content received'))

    except requests.exceptions.RequestException as e:
        print(f"\nERROR al conectar con el servidor LLM o recibir respuesta: {e}")
        print("Por favor, verifica lo siguiente:")
        print("1. Que el servidor llama.cpp esté realmente funcionando (revisa los logs).")
        print("2. Que la URL del túnel (ngrok) sea accesible desde tu navegador.")
        print("3. Puede que el servicio de ngrok haya tenido un problema o el túnel haya expirado.")
    except Exception as e:
        print(f"Ocurrió un error inesperado durante la prueba: {e}")

print("\n--- Proceso completado ---")

Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelo ya presente en Drive.
Enlace simbólico creado: /tmp/qwen2.5-7b-instruct-q4_k_m.gguf -> /content/drive/MyDrive/Colab Notebooks/llm_setup/qwen2.5-7b-instruct-q4_k_m.gguf
Ejecutable llama-server encontrado en Drive. Copiando a entorno local...

--- Iniciando Servidor Llama.cpp ---
Iniciando llama-server en segundo plano con nohup. Logs en: /tmp/llama_logs/llama-server.log
Esperando a que el servidor llama.cpp esté listo (hasta 180 segundos)... 

Servidor llama.cpp listo y escuchando en el puerto 8889.

--- Configurando Ngrok ---
Autenticando ngrok...
Creando túnel ngrok para el puerto 8889...

¡Túnel ngrok establecido con éxito! Tu URL pública es: https://0f16-35-240-132-112.ngrok-free.app

--- Realizando solicitud de prueba al servidor LLM ---
Intentando conectar al servidor LLM a través de: https://0f16-35-240-132-112.ngrok-free